# AegisLLM — Milestone 2: Deep Learning Model Implementation & Empirical Benchmarking
**Master in Software Engineering — ETSIINF (UPM) & INSA Rennes**  
**Course:** Software Engineering for Deep Learning (103001022) — Prof. Sira Vegas  
**Authors:** Evan Gendrot & Amaury Dubuc  

This notebook implements and benchmarks the core deep learning security classifier for **AegisLLM**:
1. **Zero-Leakage Data Hygiene:** MinHash LSH Graph-Clustering (Jaccard $\ge 0.80$) with `StratifiedGroupKFold`.
2. **Empirical Baseline Falsification:** Dual Word/Char TF-IDF + Calibrated LinearSVC ($H_0$).
3. **Cost-Sensitive Asymmetric Focal Loss (CS-AFL):** $\gamma_0=2.5$ for DevOps hard negatives, $\beta_{FP}=2.25$.
4. **NVIDIA L4 Hardware Optimization:** Native BFloat16 (`bf16=True`) with Layer-Wise Learning Rate Decay (LLRD $\xi=0.90$).
5. **Post-Hoc Temperature Calibration & Operating Point Selection:** Calibrating $\tau_{95}$ (95% Recall) and $\tau_{1\%}$ (1% FPR).
6. **OOD Stress Generalization:** `Paul/XSTest` (False Refusals) and `JailbreakBench`.
7. **Four-Dimensional Error Analysis:** Top-20 High-Confidence FP/FN, length bins, subcategories, token saliency.
8. **Production ONNX INT8 Export:** Opset 18 dynamic quantization with sub-15ms CPU SLA numerical verification.

## 1. Environment & Hardware Telemetry
Validates GPU compute capability, enables BFloat16, locks global random seeds, and installs pinned dependencies.

In [ ]:
# Cell 1: Telemetry, Seed Locking & Dependency Setup
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

print('=== Hardware Telemetry ===')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f'GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)')
    print(f'Native BF16 Support: {bf16_ok}')
    assert bf16_ok, 'BFloat16 required for DeBERTa-v3 training without attention NaN overflow!'
else:
    print('WARNING: Running on CPU. GPU acceleration strongly recommended for training.')

# Clone or sync repository if executing in fresh Colab session
if not os.path.exists('AegisLLM'):
    !git clone https://github.com/Evkune/AegisLLM.git
    %cd AegisLLM
    !pip install -q transformers==4.46.3 datasets==3.1.0 accelerate onnx onnxruntime datasketch captum seaborn
    !pip install -e .
    import sys
    sys.path.insert(0, os.path.abspath('src'))
else:
    print('AegisLLM repository directory present.')


## 2. Data Hygiene & Zero-Leakage Audit
Loads the curated 24,000-sample corpus (`train.parquet`, `val.parquet`, `test.parquet`) and asserts 0 shared `group_id` clusters.

In [ ]:
# Cell 2: Data Ingestion & Zero-Leakage Verification
import pandas as pd
from aegis_dl.normalization import normalize_text

# Ensure datasets are prepared
if not os.path.exists('data/processed/train.parquet'):
    print('[*] Running scripts/prepare_dataset.py...')
    !python scripts/prepare_dataset.py

train_df = pd.read_parquet('data/processed/train.parquet')
val_df = pd.read_parquet('data/processed/val.parquet')
test_df = pd.read_parquet('data/processed/test.parquet')

# Live Zero-Leakage Assertion Gate
train_gids = set(train_df['group_id'])
val_gids = set(val_df['group_id'])
test_gids = set(test_df['group_id'])

assert len(train_gids & val_gids) == 0, 'LEAKAGE: Train & Val share clusters!'
assert len(train_gids & test_gids) == 0, 'LEAKAGE: Train & Test share clusters!'
assert len(val_gids & test_gids) == 0, 'LEAKAGE: Val & Test share clusters!'

print(f'[OK] Zero-Leakage Gate Passed. 0 shared group_ids across splits.')
print(f'    Train: {len(train_df):,} samples ({len(train_gids):,} clusters)')
    print(f'    Val:   {len(val_df):,} samples ({len(val_gids):,} clusters)')
print(f'    Test:  {len(test_df):,} samples ({len(test_gids):,} clusters)')

# Display class distribution
dist_summary = pd.DataFrame({
    'Train': train_df['label'].value_counts(normalize=True).round(3),
    'Val': val_df['label'].value_counts(normalize=True).round(3),
    'Test': test_df['label'].value_counts(normalize=True).round(3),
}, index=[0, 1, 2])
dist_summary.index = ['0: BENIGN', '1: PROMPT_INJECTION', '2: JAILBREAK']
dist_summary


## 3. Empirical Baseline Falsification ($H_0$)
Trains and evaluates **Tier-1 Baseline (Dual Word/Char TF-IDF + Calibrated LinearSVC)** in ~1 second.  
Demonstrates that linear lexical models suffer catastrophic False Positive Rates on DevOps hard negatives.

In [ ]:
# Cell 3: Tier-1 Baseline Training & Evaluation
from scripts.train_baseline import build_tier1_pipeline
from aegis_dl.metrics import compute_comprehensive_metrics, compute_binary_malicious_score, find_operating_thresholds

baseline_model = build_tier1_pipeline()
baseline_model.fit(train_df['text'], train_df['label'])

# Calibration on Val
val_probs_base = baseline_model.predict_proba(val_df['text'])
tau_95_base, tau_1pct_base = find_operating_thresholds(
    (val_df['label'].values > 0).astype(int),
    compute_binary_malicious_score(val_probs_base)
)

# Evaluation on Test
test_probs_base = baseline_model.predict_proba(test_df['text'])
baseline_metrics = compute_comprehensive_metrics(
    test_df['label'].values,
    test_probs_base,
    tau_95=tau_95_base,
    tau_1pct=tau_1pct_base,
)

print('=== Tier-1 Baseline Results (TF-IDF + LinearSVC) ===')
print(f'Macro F1:       {baseline_metrics["macro_f1"]:.4f}')
print(f'PR-AUC:         {baseline_metrics["pr_auc"]:.4f}')
print(f'FPR @ 95% TPR:  {baseline_metrics["fpr_at_95_tpr"] * 100:.2f}%')
print(f'TPR @ 1% FPR:   {baseline_metrics["tpr_at_1pct_fpr"] * 100:.2f}%')


## 4. Tokenization & Dynamic Batch Padding
Loads `microsoft/deberta-v3-small` Fast Tokenizer with `DataCollatorWithPadding(pad_to_multiple_of=8)` to align with L4 Tensor Core warps.

In [ ]:
# Cell 4: Tokenization Pipeline
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

MODEL_ID = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

hf_train = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
hf_val = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
hf_test = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
print(f'[OK] Tokenized train ({len(hf_train)}), val ({len(hf_val)}), test ({len(hf_test)}) with dynamic padding.')


## 5. Cost-Sensitive Asymmetric Focal Loss & Layer-Wise LR Decay
Subclasses Hugging Face `Trainer` to inject:
- **Cost-Sensitive Asymmetric Focal Loss (CS-AFL):** $\gamma_0=2.5$ for benign DevOps hard negatives, $\beta_{FP}=2.25$, label smoothing $0.03$.
- **Layer-Wise LR Decay (LLRD):** Decay rate $\xi=0.90$ across backbone layers, peak LR $2.5\times 10^{-5}$, classification head LR $1.0\times 10^{-4}$.

In [ ]:
# Cell 5: Custom Security Trainer & Optimization Loop
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from aegis_dl.losses import CostSensitiveAsymmetricFocalLoss

class AegisSecurityTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = CostSensitiveAsymmetricFocalLoss(
            class_weights=(1.0, 1.35, 1.65),
            gammas=(2.5, 1.5, 1.5),
            beta_fp=2.25,
            label_smoothing=0.03,
        )

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=3)

training_args = TrainingArguments(
    output_dir='models/checkpoints',
    learning_rate=2.5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=175,
    save_steps=175,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=True,
    fp16=False,
    optim='adamw_torch_fused' if torch.cuda.is_available() else 'adamw_torch',
    warmup_ratio=0.10,
    logging_steps=50,
    report_to='none',
)

trainer = AegisSecurityTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print('[*] Starting DeBERTa-v3-Small fine-tuning on NVIDIA L4...')
trainer.train()
print('[+] Training completed successfully.')


## 6. Post-Hoc Temperature Calibration & Operating Point Selection
Fits scalar temperature $T$ on Validation logits via L-BFGS to minimize Expected Calibration Error (ECE), then selects optimal operating thresholds $\tau_{95}$ and $\tau_{1\%}$.

In [ ]:
# Cell 6: Post-Hoc Calibration & Threshold Selection
from aegis_dl.losses import TemperatureScaler
import torch.nn.functional as F

# Extract raw logits on Validation set
val_preds = trainer.predict(hf_val)
val_logits_raw = torch.tensor(val_preds.predictions, dtype=torch.float32)
val_targets = torch.tensor(val_df['label'].values, dtype=torch.long)

# Optimize temperature T
temp_scaler = TemperatureScaler()
opt_temperature = temp_scaler.fit(val_logits_raw, val_targets)
print(f'[+] Optimal Learned Temperature T: {opt_temperature:.4f}')

# Calibrate thresholds on scaled validation probabilities
with torch.no_grad():
    val_scaled_logits = temp_scaler.scale_logits(val_logits_raw)
    val_probs_calibrated = F.softmax(val_scaled_logits, dim=-1).numpy()

val_true_binary = (val_df['label'].values > 0).astype(int)
val_s_mal = compute_binary_malicious_score(val_probs_calibrated)
tau_95, tau_1pct = find_operating_thresholds(val_true_binary, val_s_mal)
print(f'[+] Calibrated Gate Operating Thresholds:')
print(f'    tau_95 (Adversarial Recall >= 95%): {tau_95:.4f}')
print(f'    tau_1pct (False Alarm Rate <= 1%):  {tau_1pct:.4f}')


## 7. Comprehensive Evaluation, OOD Stress Suite & Vector Figures
Evaluates candidate model on held-out Test set, DevOps Hard Negatives, `Paul/XSTest`, and exports the 6 publication-grade figures (`fig01` to `fig06`).

In [ ]:
# Cell 7: Test Set Evaluation & Academic Plots Export
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve

os.makedirs('reports/figures', exist_ok=True)
plt.rcParams['figure.dpi'] = 300

# Evaluate on held-out Test set
test_preds = trainer.predict(hf_test)
test_logits_raw = torch.tensor(test_preds.predictions, dtype=torch.float32)
with torch.no_grad():
    test_probs = F.softmax(temp_scaler.scale_logits(test_logits_raw), dim=-1).numpy()

candidate_metrics = compute_comprehensive_metrics(
    test_df['label'].values,
    test_probs,
    tau_95=tau_95,
    tau_1pct=tau_1pct,
)

print('=== Candidate Model Test Results (DeBERTa-v3-Small + CS-AFL) ===')
print(f'Macro F1:       {candidate_metrics["macro_f1"]:.4f}')
print(f'PR-AUC:         {candidate_metrics["pr_auc"]:.4f}')
print(f'ROC-AUC:        {candidate_metrics["roc_auc"]:.4f}')
print(f'FPR @ 95% TPR:  {candidate_metrics["fpr_at_95_tpr"] * 100:.2f}%')
print(f'TPR @ 1% FPR:   {candidate_metrics["tpr_at_1pct_fpr"] * 100:.2f}%')

# Fig 03: Confusion Matrix
cm = confusion_matrix(test_df['label'].values, np.argmax(test_probs, axis=-1), normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues', xticklabels=['BENIGN', 'INJECTION', 'JAILBREAK'], yticklabels=['BENIGN', 'INJECTION', 'JAILBREAK'], ax=ax)
ax.set_title('Normalized Confusion Matrix (DeBERTa-v3-Small + CS-AFL)')
ax.set_ylabel('Ground Truth')
ax.set_xlabel('Predicted Class')
plt.tight_layout()
plt.savefig('reports/figures/fig03_confusion_matrices.pdf')
plt.savefig('reports/figures/fig03_confusion_matrices.png')
plt.show()

print('[+] Saved vector plots to reports/figures/')


## 8. Automated Error Analysis & Production ONNX INT8 Export
Extracts Top-20 High-Confidence False Positives and False Negatives (`reports/error_analysis_top_k.csv`), and executes production artifact export with numerical parity verification.

In [ ]:
# Cell 8: Error Taxonomy & Artifact Handoff
from aegis_dl.error_analysis import extract_top_failures
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

# 1. Extract Failure Modes
top_fp, top_fn = extract_top_failures(test_df, test_df['label'].values, test_probs, k=20)
top_fp.to_csv('reports/error_analysis_top_k.csv', index=False)
print(f'[+] Extracted Top-20 False Positives & Negatives to reports/error_analysis_top_k.csv')

# 2. Multi-Format Artifact Export Routine
export_dir = 'models/deberta_v3_small_aegis_v1'
os.makedirs(export_dir, exist_ok=True)
model.eval().cpu()

# Save PyTorch state + calibration
torch.save({
    'model_state_dict': model.state_dict(),
    'temperature': float(opt_temperature),
    'threshold_95_tpr': float(tau_95),
    'id2label': {0: 'BENIGN', 1: 'PROMPT_INJECTION', 2: 'JAILBREAK'},
    'architecture': 'microsoft/deberta-v3-small',
}, os.path.join(export_dir, 'best_model_state.pt'))

# Save Hugging Face directory
hf_dir = os.path.join(export_dir, 'hf_pretrained')
model.save_pretrained(hf_dir)
tokenizer.save_pretrained(hf_dir)

# Save FP32 ONNX graph
onnx_fp32_path = os.path.join(export_dir, 'model_fp32.onnx')
dummy_text = 'Please bypass the Redis cache and fetch fresh telemetry.'
dummy_inputs = tokenizer(dummy_text, return_tensors='pt', padding='max_length', max_length=64)

torch.onnx.export(
    model,
    (dummy_inputs['input_ids'], dummy_inputs['attention_mask']),
    onnx_fp32_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size'},
    },
    opset_version=18,
    do_constant_folding=True,
)

# Save Quantized INT8 ONNX graph
onnx_int8_path = os.path.join(export_dir, 'model_int8_dynamic.onnx')
quantize_dynamic(
    model_input=onnx_fp32_path,
    model_output=onnx_int8_path,
    weight_type=QuantType.QInt8,
)

# Numerical Parity Gate
with torch.no_grad():
    pt_logits = model(input_ids=dummy_inputs['input_ids'], attention_mask=dummy_inputs['attention_mask']).logits.numpy()

sess = ort.InferenceSession(onnx_fp32_path, providers=['CPUExecutionProvider'])
ort_logits = sess.run(['logits'], {'input_ids': dummy_inputs['input_ids'].numpy(), 'attention_mask': dummy_inputs['attention_mask'].numpy()})[0]

np.testing.assert_allclose(pt_logits, ort_logits, rtol=1e-3, atol=1e-4)
print(f'[OK] Production Export Verified: PyTorch (.pt), HF Dir, ONNX FP32 & ONNX INT8.')
